In [1]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("imdb")

train_df = pd.DataFrame(dataset["train"])
test_df  = pd.DataFrame(dataset["test"])

print(train_df.head())
print(train_df["label"].value_counts())
print(f"Avg review length: {train_df['text'].str.split().str.len().mean():.0f} words")

README.md: 0.00B [00:00, ?B/s]

c:\Users\yazhini\Sentiment_analysis\venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yazhini\.cache\huggingface\hub\datasets--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. 

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

...

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0
2  If only to avoid making this type of film in t...      0
3  This film was probably inspired by Godard's Ma...      0
4  Oh, brother...after hearing about this ridicul...      0
label
0    12500
1    12500
Name: count, dtype: int64
Avg review length: 234 words


In [2]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)        # remove HTML tags
    text = re.sub(r"[^a-zA-Z\s]", " ", text) # remove punctuation/numbers
    text = re.sub(r"\s+", " ", text).strip() # collapse whitespace

    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return " ".join(tokens)

train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"]  = test_df["text"].apply(clean_text)

print("BEFORE:\n", train_df["text"][0][:300])
print("\nAFTER:\n", train_df["clean_text"][0][:300])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yazhini\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\yazhini\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


BEFORE:
 I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h

AFTER:
 rented curious yellow video store controversy surrounded first released also heard first seized u custom ever tried enter country therefore fan film considered controversial really see plot centered around young swedish drama student named lena want learn everything life particular want focus attent


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

X_train = vectorizer.fit_transform(train_df["clean_text"])
X_test  = vectorizer.transform(test_df["clean_text"])

y_train = train_df["label"]
y_test  = test_df["label"]

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print(classification_report(y_test, preds, target_names=["Negative", "Positive"]))

FileNotFoundError: [Errno 2] No such file or directory: 'models/tfidf_model.pkl'

In [11]:
import os
import pickle

# Always go to project root
BASE_DIR = ".."   # because notebook is inside notebooks/

model_path = os.path.join(BASE_DIR, "models", "tfidf_model.pkl")
vec_path   = os.path.join(BASE_DIR, "models", "tfidf_vectorizer.pkl")

os.makedirs(os.path.dirname(model_path), exist_ok=True)

with open(model_path, "wb") as f:
    pickle.dump(model, f)

with open(vec_path, "wb") as f:
    pickle.dump(vectorizer, f)

print("Saved in correct models folder!")

Saved in correct models folder!


In [12]:
BASE_DIR = ".."

with open(os.path.join(BASE_DIR, "models", "tfidf_model.pkl"), "rb") as f:
    tfidf_model = pickle.load(f)

with open(os.path.join(BASE_DIR, "models", "tfidf_vectorizer.pkl"), "rb") as f:
    tfidf_vec = pickle.load(f)

In [4]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()
coefs = model.coef_[0]

top_positive = feature_names[np.argsort(coefs)[-20:]]
top_negative = feature_names[np.argsort(coefs)[:20]]

print("Top positive words:", top_positive)
print("Top negative words:", top_negative)

Top positive words: ['beautiful' 'fantastic' 'must see' 'highly' 'love' 'enjoyed' 'one best'
 'superb' 'brilliant' 'today' 'well' 'fun' 'loved' 'favorite' 'amazing'
 'best' 'wonderful' 'perfect' 'excellent' 'great']
Top negative words: ['worst' 'bad' 'awful' 'waste' 'boring' 'poor' 'nothing' 'worse'
 'terrible' 'horrible' 'poorly' 'unfortunately' 'dull' 'disappointment'
 'disappointing' 'annoying' 'supposed' 'stupid' 'ridiculous' 'fails']


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

MAX_LEN   = 256
VOCAB_SIZE = 20000
EMBED_DIM  = 128
HIDDEN_DIM = 64
BATCH_SIZE = 64

from collections import Counter

def build_vocab(texts, max_vocab=VOCAB_SIZE):
    counter = Counter()
    for t in texts:
        counter.update(t.split())
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, _ in counter.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_df["clean_text"])

def encode(text, vocab, max_len=MAX_LEN):
    tokens = text.split()[:max_len]
    ids = [vocab.get(t, 1) for t in tokens]
    ids += [0] * (max_len - len(ids))  # pad
    return ids

class ReviewDataset(Dataset):
    def __init__(self, df, vocab):
        self.encodings = [encode(t, vocab) for t in df["clean_text"]]
        self.labels    = list(df["label"])

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        return (torch.tensor(self.encodings[i], dtype=torch.long),
                torch.tensor(self.labels[i],    dtype=torch.float))

train_ds = ReviewDataset(train_df, vocab)
test_ds  = ReviewDataset(test_df,  vocab)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

In [7]:
class SentimentLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        self.lstm  = nn.LSTM(EMBED_DIM, HIDDEN_DIM, batch_first=True,
                             num_layers=2, dropout=0.3, bidirectional=True)
        self.drop  = nn.Dropout(0.4)
        self.fc    = nn.Linear(HIDDEN_DIM * 2, 1)

    def forward(self, x):
        x   = self.embed(x)              # (batch, seq, embed_dim)
        out, _ = self.lstm(x)            # (batch, seq, hidden*2)
        out = torch.mean(out, dim=1)            # take last timestep
        out = self.drop(out)
        return self.fc(out).squeeze(1)   # (batch,)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_lstm = SentimentLSTM().to(device)
optimizer  = torch.optim.Adam(model_lstm.parameters(), lr=1e-3)
criterion  = nn.BCEWithLogitsLoss()

def train_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss  = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += ((preds > 0) == yb.bool()).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

for epoch in range(10):
    loss, acc = train_epoch(model_lstm, train_loader)
    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Acc: {acc:.4f}")

def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            preds = (torch.sigmoid(preds) > 0.5)
            correct += (preds == yb.bool()).sum().item()
    return correct / len(loader.dataset)

print("Test Accuracy:", evaluate(model_lstm, test_loader))

Epoch 1 | Loss: 0.5264 | Acc: 0.7296
Epoch 2 | Loss: 0.3423 | Acc: 0.8544
Epoch 3 | Loss: 0.2682 | Acc: 0.8909
Epoch 4 | Loss: 0.2172 | Acc: 0.9162
Epoch 5 | Loss: 0.1699 | Acc: 0.9359
Epoch 6 | Loss: 0.1224 | Acc: 0.9583
Epoch 7 | Loss: 0.0899 | Acc: 0.9721
Epoch 8 | Loss: 0.0674 | Acc: 0.9784
Epoch 9 | Loss: 0.0476 | Acc: 0.9851
Epoch 10 | Loss: 0.0367 | Acc: 0.9893
Test Accuracy: 0.8394


In [14]:
import pickle
import os
from transformers import pipeline

BASE_DIR = ".."   # go one level up from notebooks/

# Load TF-IDF model
tfidf_model = pickle.load(open(os.path.join(BASE_DIR, "models", "tfidf_model.pkl"), "rb"))
tfidf_vec   = pickle.load(open(os.path.join(BASE_DIR, "models", "tfidf_vectorizer.pkl"), "rb"))

# Load BERT
bert_pipe = pipeline("sentiment-analysis")

def predict_all(text):
    cleaned = clean_text(text)

    # TF-IDF prediction
    tfidf_pred  = tfidf_model.predict(tfidf_vec.transform([cleaned]))[0]
    tfidf_label = "POSITIVE" if tfidf_pred == 1 else "NEGATIVE"

    # BERT prediction
    bert_result = bert_pipe(text)[0]

    print(f"\nText   : {text[:80]}...")
    print(f"TF-IDF : {tfidf_label}")
    print(f"BERT   : {bert_result['label']} ({bert_result['score']:.2%})")
    print("-" * 50)

# Test
predict_all("This movie was absolutely fantastic, I loved every second!")
predict_all("Complete waste of time. Terrible acting and boring plot.")
predict_all("It was okay, nothing special but not terrible either.")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Users\yazhini\Sentiment_analysis\venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yazhini\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is n

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu



Text   : This movie was absolutely fantastic, I loved every second!...
TF-IDF : POSITIVE
BERT   : POSITIVE (99.99%)
--------------------------------------------------

Text   : Complete waste of time. Terrible acting and boring plot....
TF-IDF : NEGATIVE
BERT   : NEGATIVE (99.98%)
--------------------------------------------------

Text   : It was okay, nothing special but not terrible either....
TF-IDF : NEGATIVE
BERT   : POSITIVE (91.59%)
--------------------------------------------------


In [15]:
import pickle
import os
import torch
from transformers import pipeline

# -------------------------------
# Load TF-IDF model
# -------------------------------
BASE_DIR = ".."  # because notebook is inside notebooks/

with open(os.path.join(BASE_DIR, "models", "tfidf_model.pkl"), "rb") as f:
    tfidf_model = pickle.load(f)

with open(os.path.join(BASE_DIR, "models", "tfidf_vectorizer.pkl"), "rb") as f:
    tfidf_vec = pickle.load(f)

# -------------------------------
# Load BERT
# -------------------------------
bert_pipe = pipeline("sentiment-analysis")

# -------------------------------
# Prepare LSTM for inference
# -------------------------------
model_lstm.eval()

def predict_lstm(text):
    encoded = encode(clean_text(text), vocab)
    tensor = torch.tensor([encoded], dtype=torch.long).to(device)

    with torch.no_grad():
        output = model_lstm(tensor)
        prob = torch.sigmoid(output).item()

    label = "POSITIVE" if prob > 0.5 else "NEGATIVE"
    return label, prob

# -------------------------------
# Combined prediction function
# -------------------------------
def predict_all(text):
    cleaned = clean_text(text)

    # TF-IDF prediction
    tfidf_pred  = tfidf_model.predict(tfidf_vec.transform([cleaned]))[0]
    tfidf_label = "POSITIVE" if tfidf_pred == 1 else "NEGATIVE"

    # LSTM prediction
    lstm_label, lstm_prob = predict_lstm(text)

    # BERT prediction
    bert_result = bert_pipe(text)[0]

    print(f"\nText   : {text[:80]}...")
    print(f"TF-IDF : {tfidf_label}")
    print(f"LSTM   : {lstm_label} ({lstm_prob:.2%})")
    print(f"BERT   : {bert_result['label']} ({bert_result['score']:.2%})")
    print("-" * 50)

# -------------------------------
# Test examples
# -------------------------------
predict_all("This movie was absolutely fantastic, I loved every second!")
predict_all("Complete waste of time. Terrible acting and boring plot.")
predict_all("It was okay, nothing special but not terrible either.")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu



Text   : This movie was absolutely fantastic, I loved every second!...
TF-IDF : POSITIVE
LSTM   : POSITIVE (99.69%)
BERT   : POSITIVE (99.99%)
--------------------------------------------------

Text   : Complete waste of time. Terrible acting and boring plot....
TF-IDF : NEGATIVE
LSTM   : NEGATIVE (0.11%)
BERT   : NEGATIVE (99.98%)
--------------------------------------------------

Text   : It was okay, nothing special but not terrible either....
TF-IDF : NEGATIVE
LSTM   : NEGATIVE (2.76%)
BERT   : POSITIVE (91.59%)
--------------------------------------------------


In [17]:
import torch
import os

BASE_DIR = ".."  # go back to project root

model_path = os.path.join(BASE_DIR, "models", "lstm_model.pth")

os.makedirs(os.path.dirname(model_path), exist_ok=True)

torch.save(model_lstm.state_dict(), model_path)

print("✅ LSTM model saved successfully!")

✅ LSTM model saved successfully!


In [18]:
import pickle

with open(os.path.join(BASE_DIR, "models", "vocab.pkl"), "wb") as f:
    pickle.dump(vocab, f)